In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sbn
import datetime as dt
import math
import random 
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder
from itertools import product

In [2]:
# %load_ext tensorboard
%matplotlib inline
sbn.set_style("whitegrid")

### Load the data and generate the dataset for modeling

In [3]:
transaction = pd.read_csv('../final_data/transaction.csv')
transaction.head(1)

,transaction_id,product_id,customer_id,first_name,last_name,gender,num_prev_purchase,DOB,age,job_title,...,product_size,list_price,standard_cost,transaction_date,day_of_week,year,month,day,online_order,order_status
0,1,2,2950,Kristos,Anthony,Male,19,1955-01-11,68.0,Software Engineer I,...,medium,71.49,53.62,2017-02-25,Saturday,2017,2,25,False,Approved


In [4]:
transaction.columns

Index(['transaction_id', 'product_id', 'customer_id', 'first_name',
       'last_name', 'gender', 'num_prev_purchase', 'DOB', 'age', 'job_title',
       'job_industry', 'wealth_segment', 'owns_car', 'street', 'state',
       'country', 'post_code', 'brand', 'product_line', 'product_class',
       'product_size', 'list_price', 'standard_cost', 'transaction_date',
       'day_of_week', 'year', 'month', 'day', 'online_order', 'order_status'],
      dtype='object')

In [5]:
customer_info = ['customer_id', 'gender', 'num_prev_purchase', 'age', 'job_industry', 'wealth_segment', 'owns_car', 'state']
product_info = ['product_id', 'brand', 'product_line', 'product_class', 'product_size', 'online_order']

In [6]:
# create all possible combination of customer and product id
all_possible_comb = pd.DataFrame(product(transaction['customer_id'].unique(), transaction['product_id'].unique()))
all_possible_comb.rename(columns={0: "customer_id", 1: "product_id"},inplace=True)

In [7]:
# deriving rating based on quantity, price, and transaction_date
transaction['transaction_date'] = pd.to_datetime(transaction['transaction_date'])
snapshot_date = transaction['transaction_date'].min() - dt.timedelta(days=1)
user_item_table = transaction.groupby(['customer_id','product_id']).agg({
    'transaction_id': 'count',
    'list_price': 'first',
    'transaction_date': lambda x: (x.max()-snapshot_date).days
}).reset_index()
user_item_table.rename(columns={'transaction_id': 'quantity', 'list_price': 'price', 'transaction_date': 'day'}, inplace=True)
user_item_table.head(1)

,customer_id,product_id,quantity,price,day
0,1,2,1,71.49,52


In [8]:
user_item_table.shape

(18819, 5)

In [9]:
# merge with all possible options 
user_item_table = pd.merge(user_item_table,all_possible_comb, on=['customer_id','product_id'], how='outer')
user_item_table = user_item_table.fillna(0)

In [10]:
user_item_table.shape

(352389, 5)

In [11]:
# merge the rating table with customer and product features
user_item_table = pd.merge(user_item_table,transaction[product_info].groupby(['product_id']).first().reset_index(), on=['product_id'], how='left')
user_item_table = pd.merge(user_item_table,transaction[customer_info].groupby(['customer_id']).first().reset_index(), on=['customer_id'], how='left')
user_item_table.loc[(user_item_table.job_industry.isnull()).values, 'job_industry'] = 'Others'
user_item_table.dropna(inplace=True)
user_item_table.reset_index(inplace=True,drop=True)
user_item_table.head(3)

,customer_id,product_id,quantity,price,day,brand,product_line,product_class,product_size,online_order,gender,num_prev_purchase,age,job_industry,wealth_segment,owns_car,state
0,1,2,1.0,71.49,52.0,Solex,Standard,medium,medium,False,Female,93,69.0,Health,Mass Customer,N,NSW
1,1,9,1.0,742.54,343.0,OHM Cycles,Road,medium,medium,True,Female,93,69.0,Health,Mass Customer,N,NSW
2,1,11,1.0,1274.93,88.0,Giant Bicycles,Standard,high,medium,False,Female,93,69.0,Health,Mass Customer,N,NSW


In [12]:
# Normalization
user_item_table['quantity_norm'] = pd.Series(MinMaxScaler((0,1)).fit_transform(np.array(user_item_table['quantity']).reshape(-1, 1)).reshape(-1))
user_item_table['day_norm'] = pd.Series(MinMaxScaler().fit_transform(np.array(user_item_table['day']).reshape(-1, 1)).reshape(-1))

price = user_item_table['price']
listMax = price.max()
price_norm = [i/listMax for i in price]
price_norm = [1/(1+math.exp(-i)) for i in price_norm]
user_item_table['price_norm'] = price_norm

user_item_table.head(10)

,customer_id,product_id,quantity,price,day,brand,product_line,product_class,product_size,online_order,gender,num_prev_purchase,age,job_industry,wealth_segment,owns_car,state,quantity_norm,day_norm,price_norm
0,1,2,1.0,71.49,52.0,Solex,Standard,medium,medium,False,Female,93,69.0,Health,Mass Customer,N,NSW,0.2,0.142857,0.508545
1,1,9,1.0,742.54,343.0,OHM Cycles,Road,medium,medium,True,Female,93,69.0,Health,Mass Customer,N,NSW,0.2,0.942308,0.587837
2,1,11,1.0,1274.93,88.0,Giant Bicycles,Standard,high,medium,False,Female,93,69.0,Health,Mass Customer,N,NSW,0.2,0.241758,0.647846
3,1,23,1.0,688.63,86.0,Norco Bicycles,Mountain,low,small,True,Female,93,69.0,Health,Mass Customer,N,NSW,0.2,0.236264,0.581578
4,1,25,1.0,1538.99,139.0,Giant Bicycles,Road,medium,medium,True,Female,93,69.0,Health,Mass Customer,N,NSW,0.2,0.381868,0.676086
5,1,31,1.0,752.64,348.0,WeareA2B,Standard,medium,medium,False,Female,93,69.0,Health,Mass Customer,N,NSW,0.2,0.956044,0.589007
6,1,32,1.0,642.70,155.0,Giant Bicycles,Standard,medium,medium,False,Female,93,69.0,Health,Mass Customer,N,NSW,0.2,0.425824,0.576225
7,1,38,1.0,2091.47,96.0,Trek Bicycles,Standard,medium,large,True,Female,93,69.0,Health,Mass Customer,N,NSW,0.2,0.263736,0.731059
8,1,47,1.0,1720.70,131.0,Trek Bicycles,Road,low,small,False,Female,93,69.0,Health,Mass Customer,N,NSW,0.2,0.359890,0.694814
9,1,72,1.0,360.40,5.0,Norco Bicycles,Standard,medium,medium,True,Female,93,69.0,Health,Mass Customer,N,NSW,0.2,0.013736,0.542973


In [13]:
# standardize other features and encode categorical features
prev_purchase_scaler = MinMaxScaler(feature_range=(0,1))
age_scaler = MinMaxScaler(feature_range=(0,1))
user_item_table['num_prev_purchase'] = pd.Series(prev_purchase_scaler.fit_transform(np.array(user_item_table['num_prev_purchase']).reshape(-1, 1)).reshape(-1))
user_item_table['age'] = pd.Series(age_scaler.fit_transform(np.array(user_item_table['age']).reshape(-1, 1)).reshape(-1))

# label encoding
gender_encoder = LabelEncoder()
user_item_table['gender'] = gender_encoder.fit_transform(user_item_table.gender)
owns_car_encoder = LabelEncoder()
user_item_table['owns_car'] = gender_encoder.fit_transform(user_item_table.owns_car)
online_order_encoder = LabelEncoder()
user_item_table['online_order'] = gender_encoder.fit_transform(user_item_table.online_order)

# one-hot encoding
for i in ['job_industry', 'wealth_segment', 'state', 'brand', 'product_line', 'product_class', 'product_size']:
    user_item_table = pd.concat([user_item_table, pd.get_dummies(user_item_table[i], prefix=i)],axis=1)
    user_item_table.drop(i, axis=1, inplace=True)
user_item_table.head()

,customer_id,product_id,quantity,price,day,online_order,gender,num_prev_purchase,age,owns_car,...,product_line_Mountain,product_line_Road,product_line_Standard,product_line_Touring,product_class_high,product_class_low,product_class_medium,product_size_large,product_size_medium,product_size_small
0,1,2,1.0,71.49,52.0,0,0,0.939394,0.685714,0,...,0,0,1,0,0,0,1,0,1,0
1,1,9,1.0,742.54,343.0,1,0,0.939394,0.685714,0,...,0,1,0,0,0,0,1,0,1,0
2,1,11,1.0,1274.93,88.0,0,0,0.939394,0.685714,0,...,0,0,1,0,1,0,0,0,1,0
3,1,23,1.0,688.63,86.0,1,0,0.939394,0.685714,0,...,1,0,0,0,0,1,0,0,0,1
4,1,25,1.0,1538.99,139.0,1,0,0.939394,0.685714,0,...,0,1,0,0,0,0,1,0,1,0


In [14]:
# generate binary target
user_item_table['target'] = 0
user_item_table.loc[user_item_table.quantity!=0, 'target'] = 1

In [15]:
user_item_table.target.value_counts()

0    326224
1     18388
Name: target, dtype: int64

In [16]:
user_item_table.quantity.value_counts()

0.0    326224
1.0     17493
2.0       729
3.0       116
4.0        36
5.0        14
Name: quantity, dtype: int64

In [17]:
user_item_table.sort_values(by=['customer_id','product_id'],inplace=True)

In [18]:
le_users = LabelEncoder()
le_items = LabelEncoder()
user_item_table['customer_id'] = le_users.fit_transform(user_item_table.customer_id.to_numpy())
user_item_table['product_id'] = le_items.fit_transform(user_item_table.product_id.to_numpy())
np.save('user_encoder.npy', le_users.classes_)
np.save('item_encoder.npy', le_items.classes_)
user_item_table.head()

,customer_id,product_id,quantity,price,day,online_order,gender,num_prev_purchase,age,owns_car,...,product_line_Road,product_line_Standard,product_line_Touring,product_class_high,product_class_low,product_class_medium,product_size_large,product_size_medium,product_size_small,target
26748,0,0,0.0,0.00,0.0,0,0,0.939394,0.685714,0,...,1,0,0,0,0,1,0,1,0,0
26746,0,1,0.0,0.00,0.0,0,0,0.939394,0.685714,0,...,0,1,0,0,0,1,0,1,0,0
0,0,2,1.0,71.49,52.0,0,0,0.939394,0.685714,0,...,0,1,0,0,0,1,0,1,0,1
26725,0,3,0.0,0.00,0.0,1,0,0.939394,0.685714,0,...,0,1,0,0,0,1,1,0,0,0
26764,0,4,0.0,0.00,0.0,1,0,0.939394,0.685714,0,...,0,1,0,1,0,0,0,1,0,0


In [19]:
user_item_table.columns

Index(['customer_id', 'product_id', 'quantity', 'price', 'day', 'online_order',
       'gender', 'num_prev_purchase', 'age', 'owns_car', 'quantity_norm',
       'day_norm', 'price_norm', 'job_industry_Agriculture',
       'job_industry_Entertainment', 'job_industry_Financial Services',
       'job_industry_Health', 'job_industry_IT', 'job_industry_Manufacturing',
       'job_industry_Others', 'job_industry_Property', 'job_industry_Retail',
       'job_industry_Telecommunications', 'wealth_segment_Affluent Customer',
       'wealth_segment_High Net Worth', 'wealth_segment_Mass Customer',
       'state_NSW', 'state_QLD', 'state_VIC', 'brand_Giant Bicycles',
       'brand_Norco Bicycles', 'brand_OHM Cycles', 'brand_Solex',
       'brand_Trek Bicycles', 'brand_WeareA2B', 'product_line_Mountain',
       'product_line_Road', 'product_line_Standard', 'product_line_Touring',
       'product_class_high', 'product_class_low', 'product_class_medium',
       'product_size_large', 'product_size_med

In [20]:
user_item_table.isna().sum()

customer_id                         0
product_id                          0
quantity                            0
price                               0
day                                 0
online_order                        0
gender                              0
num_prev_purchase                   0
age                                 0
owns_car                            0
quantity_norm                       0
day_norm                            0
price_norm                          0
job_industry_Agriculture            0
job_industry_Entertainment          0
job_industry_Financial Services     0
job_industry_Health                 0
job_industry_IT                     0
job_industry_Manufacturing          0
job_industry_Others                 0
job_industry_Property               0
job_industry_Retail                 0
job_industry_Telecommunications     0
wealth_segment_Affluent Customer    0
wealth_segment_High Net Worth       0
wealth_segment_Mass Customer        0
state_NSW   

In [21]:
# save the data for modeling
user_item_table.to_csv('../final_data/user_item_table.csv',index=False)